1\. **Radioactive decay chain**

${\rm Tl}^{208}$ decays to ${\rm Pb}^{208}$ with a half-lieve of 3.052 minutes. Suppose to start with a sample of 1000 Thallium atoms and 0 of Lead atoms.

* Take steps in time of 1 second and at each time-step decide whether each Tl atom has decayed or not, accordingly to the probability $p(t)=1-2^{-t/\tau}$. Subtract the total number of Tl atoms that decayed at each step from the Tl sample and add them to the Lead one. Plot the evolution of the two sets as a function of time  
* Repeat the exercise by means of the inverse transform method: draw 1000 random numbers from the non-uniform probability distribution $p(t)=2^{-t/\tau}\frac{\ln 2}{\tau}$ to represent the times of decay of the 1000 Tl atoms. Make a plot showing the number of atoms that have not decayed as a function of time

In [ ]:
import numpy.random as npr
import numpy as np
import matplotlib.pyplot as plt

npr.seed(15)

tstep = 1#s
tau = 3052*60#s
timeobs = 5*tau#s
N0 = 1000
NPb = 0
NTl = N0
Pb = []
Tl = []

def decay_prob(x,tau = tau):
    return 1-2**(-x/tau)

for i in range(timeobs):
    if(NTl==0): break
    ndecays = ((npr.random(NTl)<decay_prob(1))*1).sum()
    #print(ndecays)
    NPb+=ndecays
    NTl-=ndecays
    Pb.append(NPb)
    Tl.append(NTl)

plt.plot(range(timeobs),Pb,range(timeobs),Tl)
plt.vlines(tau,0,1000,"k",label="Halflife")
plt.grid()
plt.legend();

### Tenere a mente che la probabilità va valutata in 1

### Second part:

In [ ]:
import numpy.random as npr
import numpy as np
import matplotlib.pyplot as plt

npr.seed(15)

tau = 3052*60#s

u = npr.random(N0)  #Sampling 1000 random numbers

def p_times(z,tau = tau):
    return -tau*np.log2(1-z)

times = np.sort(p_times(u))
N = np.arange(0,N0,1)[::-1]

plt.plot(times,N)
plt.vlines(tau,0,1000,'b',label='Halflife')
plt.legend()
plt.grid();


2\. **Rutherford Scattering**

The scattering angle $\theta$ of $\alpha$ particles hitting a positively charged nucleus of a Gold atom ($Z=79$) follows the rule:

$$
\tan{\frac{1}{2} \theta} = \frac{Z e^2} {2\pi \epsilon_0 E b}
$$

where $E=7.7$ MeV and $b$ beam is the impact parameter. The beam is represented by a 2D gaussian distribution with $\sigma=a_0/100$ for both coordinates ($a_0$ being the Bohr radius). Assume 1 million $\alpha$ particles are shot on the gold atom.

Computing the fraction of particles that "bounce back",i.e. those particle whose scattering angle is greater than $\pi/2$ (which set a condition on the impact parameter $b$)

In [ ]:
import numpy as np
import numpy.random as npr

N = int(1e6) #alfa particles
#b the impact parameter is the distance from the origin of sorted coordinates
a_0 = 5.3e-11

counts=0
def theta(b):
    Z = 79
    e = 1.602e-19
    E = 7.7e6*e
    e0 = 8.85e-12
    return Z*e**2/(2*np.pi*e0*E*b)  #restutuisce tan(theta/2)--> a pi/2 è 1

z = npr.rand(N)
b = np.sqrt(-2*(a_0/100)**2*np.log(1-z))
ang = theta(b)
counts = sum((ang>1)*1)
print(f"{(counts/N)*100:.3f}% backscattered")

3\. **Monte Carlo integration: hit/miss vs mean value method**

Consider the function 

$$f(x) =\sin^2{\frac{1}{x(2-x)}}$$

* Compute the integral of $f(x)$ between 0 and 2 with the hit/miss method. Evaluate the error of your estimate
* Repeat the integral with the mean value method. Evaluate the error and compare it with the previous one

In [ ]:
import numpy as np
import numpy.random as npr
import matplotlib.pyplot as plt

npr.seed(15)

def f(x):
    return np.sin(1/(x*(2-x)))**2

N = int(1e6) #total points number

x = np.sort(npr.rand(N)*2)   # randomly choosing coordinates in I
y = npr.rand(N)  # random values to compare the function

#plt.plot(x,f(x))

eval = np.trapezoid(f(x),x)

test = sum((y<f(x))*1)
res = test/N*2  #Interval*fraction obtained

print("Risultati con il metodo MC:",res,eval,abs(res-eval)/res)

x = npr.rand(N)*2
y = f(x) #random sorted values of the function
I = 2/N*sum(y)
print("\nRisultati mean value",I,abs(I-eval)/I)

4\. **Monte Carlo integration in high dimension**

* Start of by computing the area of a circle of unit radius, by integrating the function 

$$
f(x,y)=
\left\{
\begin{array}{ll}
      1 & x^2+y^2\le 1 \\
      0 & {\rm elsewhere}
\end{array} 
\right.
$$

* Generalize the result for a 10D sphere



In [ ]:
import numpy as np
import numpy.random as npr
import scipy.special as scis


def dncirc(data):
    res=[]
    for i in data:
        if sum(np.array(i)**2)<1:
            res.append(1)
        else:
            res.append(0)
    return np.array(res)

dim = int(input("Inserire la dimensione scelta:"))
vec = []
samplesize = int(1e6)

vol = 1

for i in range(dim):
    vec.append(npr.rand(samplesize)*2-1)


r = dncirc(np.array(vec).T)

A_ext = r.sum()/r.size*2**(dim)
A_real = np.pi**(dim/2)/scis.gamma(1+dim/2)


print(f"Il risultato ottenuto, per {dim} dimensioni è: {A_ext:.3f} con MC, {A_real:.3f} atteso.\n\nL'errore relativo è del {(abs(A_real-A_ext)/A_ext*100):.2f}%")


5\. **Monte Carlo integration with importance sampling** 

Calculate the value of the integral:

$$
I=\int_0^1 \frac{x^{-1/2}}{e^x+1} dx
$$

using the importance sampling method with $w(x)=1/\sqrt{x}$. You should get a result about 0.84

In [ ]:
import numpy.random as npr
import numpy as np

def f(x):
    return np.sqrt(x)**-1/(np.exp(x)+1)
def test_f(x):
    return np.sqrt(x)**-1

N = int(1e6)
x = npr.rand(N)**2      # devo estrarre i dati come se uscissero da 1/sqrt(x) per avere una maggiore precisione nel calcolo dell'integrale
                        # così sto prendendo più dati equispaziati una volta applicata la mia funzione--> dati vicini a 0--> integrale più preciso

int_w = 2
res = ((f(x)/test_f(x)).sum()/N)*int_w
print(f"{res:.2f}")

Using $$ \int_0^1\frac{1}{\sqrt{x}}dx=2\sqrt{x}|^1_0=2$$